In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import pickle
from helpers.predictors import *

USE_TAM = True
# Initialize models with TAM tracker
initialize_models(use_tam=USE_TAM)

data_collection = 'cha/cha_short'  # Data collection to use 

# Configuration
data_dir = f'inputs/{data_collection}'  # Base directory for inputs
detection_dir = f'outputs/{data_collection}'  # Directory containing detection txt files
show=False  # Show the video while processing
original_resolution = (3840, 2160)
# original_resolution = (1920, 1080)
resolution = (960, 540)  # Resolution of the video in case of resampling

max_hands = 2  # Maximum number of hands to track

Device: cuda
Torch compile enabled: True
Loads checkpoint by local backend from path: checkpoints/hands/detection/cascade_rcnn_x101_64x4d_fpn_20e_onehand10k-dac19597_20201030.pth
Loads checkpoint by local backend from path: checkpoints/body/detection/rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth
Disable torch compile due to unsupported GPU.
Loads checkpoint by local backend from path: checkpoints/hands/pose/rtmpose-m_simcc-hand5_pt-aic-coco_210e-256x256-74fb594_20230320.pth
Loads checkpoint by local backend from path: checkpoints/body/pose/rtmpose-x_simcc-body7_pt-body7-halpe26_700e-384x288-7fb6e239_20230606.pth


c:\Users\aelvy\miniconda3\envs\orpose\Lib\site-packages\mmengine\utils\manager.py:113: UserWarning: <class 'mmpose.visualization.local_visualizer.PoseLocalVisualizer'> instance named of visualizer has been created, the method `get_instance` should not accept any other arguments
  warnings.warn(


Models initialized with EfficientTAM tracker
Pose estimation models initialized
YOLO models initialized


In [2]:
def read_detection_file(filepath, with_confidence=False):
    """Read detection file containing frame index and bounding boxes"""
    detections = {}
    stride = 5 if with_confidence else 4
    with open(filepath, 'r') as f:
        for line in f:
            values = list(map(float, line.strip().split()))
            frame_idx = int(values[0])
            boxes = []
            # Each box has 4 coordinates
            for i in range(1, len(values), stride):
                boxes.append(values[i:i+4])

            boxes = np.array(boxes)

            if resolution != original_resolution:
                # Rescale bounding boxes to original resolution
                
                boxes[:, 0] *= resolution[0] / original_resolution[0]
                boxes[:, 1] *= resolution[1] / original_resolution[1]
                boxes[:, 2] *= resolution[0] / original_resolution[0]
                boxes[:, 3] *= resolution[1] / original_resolution[1]
            detections[frame_idx] = boxes
    return detections

def prepare_video_frames(video_path, resample_to=None):
    """Create frames directory if video file exists"""
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Video not found: {video_path}")
        
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    frames_dir = os.path.join(os.path.dirname(video_path), video_name)
    
    if not os.path.exists(frames_dir):
        os.makedirs(frames_dir)
        cap = cv2.VideoCapture(video_path)
        frame_idx = 0
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if resample_to is not None:
                frame = cv2.resize(frame, resample_to)
            cv2.imwrite(os.path.join(frames_dir, f"{frame_idx:05d}.jpg"), frame)
            frame_idx += 1
            
        cap.release()
        
    return frames_dir

In [3]:
def process_video_hands(video_name):
    """Process a single video file and track all boxes as one object"""

    def box_tracked(tracks, frame_idx, box):
        """Check if box is already tracked in the tracks"""

        box_mean_x = (box[0] + box[2]) / 2
        box_mean_y = (box[1] + box[3]) / 2
        if frame_idx not in tracks:
            return False

        for tracked_box in tracks[frame_idx].values():
            if len(tracked_box) == 0:
                continue
            tracked_box = tracked_box.flatten()
            tracked_box_mean_x = (tracked_box[0] + tracked_box[2]) / 2
            tracked_box_mean_y = (tracked_box[1] + tracked_box[3]) / 2
            tracked_in_box = (tracked_box_mean_x >= box[0] and tracked_box_mean_x <= box[2] and tracked_box_mean_y >= box[1] and tracked_box_mean_y <= box[3])
            box_in_tracked = (box_mean_x >= tracked_box[0] and box_mean_x <= tracked_box[2] and box_mean_y >= tracked_box[1] and box_mean_y <= tracked_box[3])
            if tracked_in_box and box_in_tracked:
                return True
        return False


    # Get paths
    video_path = os.path.join(data_dir, f"{video_name}.MP4")
    detection_path = os.path.join(detection_dir, f"{video_name}.txt")

    if not os.path.exists(detection_path):
        print(f"No detection file found for {video_name}")
        return None, None

    # Prepare video frames
    frames_dir = prepare_video_frames(video_path, resolution)

    # Read detections
    detections = read_detection_file(detection_path, with_confidence=True)

    # Initialize tracker for all boxes under single ID
    obj_id = 0
    all_tracks = None
    predictor = None
    inference_state = None
    frame_names = None

    # Process each frame with detections
    for frame_idx, boxes in detections.items():
        for box in boxes:
            if (all_tracks is None or not box_tracked(all_tracks, frame_idx, box)) and obj_id < max_hands:
                print(f"Frame {frame_idx}: {box}")
                print(f"Object ID: {obj_id}")
                # Initialize tracker with first frame's boxes if not done yet
                inference_state, predictor, frame_names = add_object(
                    frames_dir,
                    input_box=box,
                    frame_idx=frame_idx,
                    obj_id=obj_id,
                    show=show,
                    inference_state=inference_state,
                    predictor_in=predictor)

                # Track the object through video
                _, track_boxes = track_object(
                    frames_dir,
                    inference_state,
                    predictor,
                    frame_names,
                    show=show,
                    prev_bboxes=None,
                    whole_mask=True)

                obj_id += 1
                all_tracks = track_boxes

    if frame_names is None:
        return None, None

    return all_tracks, frame_names


In [4]:
def adjust_tracks(tracks, adj_resolution=resolution, orig_resolution=original_resolution):
    for frame_idx, frame_tracks in tracks.items():
        for obj_id, box in frame_tracks.items():
            box = box.flatten()
            if len(box) == 0:
                continue
            box[0] *= orig_resolution[0] / adj_resolution[0]
            box[1] *= orig_resolution[1] / adj_resolution[1]
            box[2] *= orig_resolution[0] / adj_resolution[0]
            box[3] *= orig_resolution[1] / adj_resolution[1]
            tracks[frame_idx][obj_id] = box
    return tracks

In [ ]:
# Process a single video
# video_name = "gopro9_synced_cut"  # Change this to process different videos
# tracks, frame_names = process_video_hands(video_name)

# # Render video with bounding boxes
# render_bbox_video(f"outputs/{data_collection}/{video_name}_tracked", os.path.join(data_dir, video_name), tracks, frame_names)

[249  28 533 391]
0.1102 0.2
0.1698 0.2
False False
[250  31 530 389]
0.0975 0.2
0.1619 0.2
False False
[252  37 530 391]


c:\Users\aelvy\miniconda3\envs\orpose\Lib\site-packages\mmdet\models\layers\se_layer.py:158: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
c:\Users\aelvy\miniconda3\envs\orpose\Lib\site-packages\mmdet\models\backbones\csp_darknet.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


0.2144 0.2
0.2776 0.2
True True
Frame 2: [     252.88      37.743       530.1      391.58]
Object ID: 0


propagate in video: 100%|██████████| 3/3 [00:00<00:00,  4.66it/s]


[263  68 529 386]
0.08734 0.2
0.1531 0.2
False False
[241  20 547 385]
0.3003 0.2
0.377 0.2
True True
Frame 10: [     241.26      20.174      547.89      385.12]
Object ID: 1


propagate in video: 100%|██████████| 3/3 [00:01<00:00,  2.34it/s]


[242  15 532 383]
0.2208 0.2
0.2947 0.2
True True
[242  14 538 381]
0.1246 0.2
0.1973 0.2
False False
[242  10 537 380]
0.1547 0.2
0.2312 0.2
True False
[244  23 535 380]
0.2534 0.2
0.303 0.2
True True
[247  23 536 378]
0.2494 0.2
0.2832 0.2
True True
[251  36 532 378]
0.2308 0.2
0.294 0.2
True True
[244  19 535 376]
0.1194 0.2
0.1802 0.2
False False
[246  20 537 375]
0.2578 0.2
0.3047 0.2
True True
[251  33 533 375]
0.11676 0.2
0.1672 0.2
False False
[246  60 542 374]
0.0722 0.2
0.1366 0.2
False False
[248  18 537 373]
0.2827 0.2
0.351 0.2
True True
[249  55 540 371]
0.0776 0.2
0.1432 0.2
False False
[249  38 540 372]
0.1647 0.2
0.2269 0.2
True False
[249  28 539 372]
0.1387 0.2
0.1943 0.2
False False
[255  67 539 372]
0.0905 0.2
0.1526 0.2
False False
[251  35 545 372]
0.198 0.2
0.2651 0.2
True False
[249  26 546 374]
0.2374 0.2
0.2842 0.2
True True
[249  22 546 375]
0.1349 0.2
0.1954 0.2
False False
[151   0 345 507]
0.1987 0.2
0.276 0.2
True False
[248  44 544 376]
0.0897 0.2
0.149

100%|██████████| 60/60 [00:33<00:00,  1.82it/s]


In [6]:
# # Process videos
for video_file in os.listdir(data_dir):
    if video_file.endswith('.MP4'):
        video_name = os.path.splitext(video_file)[0]
        print(f"Processing {video_name}...")
        tracks, frame_names = process_video_hands(video_name)
        if tracks is not None:
            print(f"Processed {video_name}: {len(tracks)} frames")

            # Render video with bounding boxes
            render_bbox_video(f"outputs/{data_collection}/{video_name}_tracked", os.path.join(data_dir, video_name), tracks, frame_names)

            with open(f"outputs/{data_collection}/tracked_bboxes_{video_name}.pkl", 'wb+') as f:
                tracks = adjust_tracks(tracks)
                pickle.dump(tracks, f)
        else:
            print(f"Failed to process {video_name}")

    


Processing gopro10_synced_cut...
Frame 17: [     473.14      146.22      524.72      201.34]
Object ID: 0


propagate in video: 100%|██████████| 18/18 [00:05<00:00,  3.54it/s]


Processed gopro10_synced_cut: 60 frames


100%|██████████| 60/60 [00:09<00:00,  6.36it/s]


Processing gopro11_synced_cut...
Frame 4: [     498.86      64.095      559.85      113.71]
Object ID: 0


...

propagate in video: 100%|██████████| 60/60 [00:18<00:00,  3.25it/s]


Frame 30: [      151.3     0.91582      345.98      507.13]
Object ID: 1


propagate in video: 100%|██████████| 60/60 [00:26<00:00,  2.26it/s]


Processed gopro9_synced_cut: 60 frames


100%|██████████| 60/60 [00:10<00:00,  5.78it/s]
